# Streamed DPO on a free Colab T4: does this run ship?

Runtime -> Change runtime type -> **T4 GPU**. Run the cells in order.

Every phase goes through `scripts/run_logged.sh`, which keeps the raw evidence:
`logs/t4/<phase>.log` (every line UTC-timestamped), `logs/t4/<phase>.exit`,
`nvidia-smi/<phase>.csv` (500 ms samples) and `nvidia-smi/<phase>_full.txt` (a full
`nvidia-smi` screen every 60 s). Nothing is filtered: failed phases stay in the logs.

## 0. Machine

In [ ]:
!git clone -q https://github.com/abdulwaarith0/soup-dpo-t4-take-home.git
%cd soup-dpo-t4-take-home
!mkdir -p logs/t4 nvidia-smi
!nvidia-smi | tee nvidia-smi/00_start.txt
!(date -u; free -h; df -h /; nproc; cat /proc/cpuinfo | grep 'model name' | head -1) | tee logs/t4/00_host.txt

## 1. Install exactly what the task says

`pip install "soup-cli[train]"` resolves to **0.75.1** (the latest release). It is pinned here so a
re-run cannot silently pick up a newer Soup; the versions of everything underneath are recorded.

In [ ]:
!pip install -q "soup-cli[train]==0.75.1" 2>&1 | tail -3
!python -c "import torch,transformers,peft,trl,accelerate,soup_cli,sys;print('python',sys.version.split()[0]);print('torch',torch.__version__,'cuda',torch.version.cuda);print('transformers',transformers.__version__);print('peft',peft.__version__);print('trl',trl.__version__);print('accelerate',accelerate.__version__);print('soup-cli',soup_cli.__version__);print('bf16 native:',torch.cuda.is_bf16_supported(including_emulation=False),'| bf16 incl. emulation:',torch.cuda.is_bf16_supported())" | tee logs/t4/01_versions.txt

## 2. Data: rebuild, audit, and run Soup's own data checks

`prepare_data.py` re-derives the committed files from the pinned dataset revision; the printed
hashes must match `data/README.md`. Then my audit (`audit_data.py`) and Soup's checks run on the
same file, so the report can say which quirks Soup catches.

In [ ]:
!bash scripts/run_logged.sh 02_prepare_data -- python data/prepare_data.py
!bash scripts/run_logged.sh 03_audit_data -- python scripts/audit_data.py --data data/train.jsonl --max-length 1024
!bash scripts/run_logged.sh 04_soup_data_lint -- soup data lint data/train.jsonl
!bash scripts/run_logged.sh 05_soup_data_validate -- soup data validate data/train.jsonl
!bash scripts/run_logged.sh 06_soup_data_doctor -- soup data doctor data/train.jsonl --model Qwen/Qwen2.5-1.5B-Instruct

## 3. Memory budget BEFORE training, then Soup's own pre-flight

The budget is printed and saved before any training step exists to compare it with.

In [ ]:
!bash scripts/run_logged.sh 07_memory_budget -- python scripts/memory_budget.py --batch 2 --seq 1024
!bash scripts/run_logged.sh 08_soup_doctor -- soup doctor
!bash scripts/run_logged.sh 09_dry_run -- soup train --config configs/dpo_stream_t4.yaml --dry-run

## 4. The run under review

Runs in the background so a browser disconnect cannot kill it; the next cell shows progress.
`--probe` records the logits' dtype/shape and allocated memory at the first policy and reference
log-softmax calls (read-only, see `scripts/train_with_mem.py`).

In [ ]:
!nohup bash scripts/run_logged.sh 10_train_main -- python scripts/train_with_mem.py --config configs/dpo_stream_t4.yaml --yes --probe > /dev/null 2>&1 &

In [ ]:
!tail -c 1500 logs/t4/10_train_main.log; echo; cat logs/t4/10_train_main.exit 2>/dev/null || echo "(still running)"

## 5. Part 2: did the run change the model, in the direction the data asked for?

Fresh process, saved adapter file, 100 pairs the run never saw, float32. Also builds and checks
the two controls (zeroed adapter, norm-matched random adapter).

In [ ]:
!bash scripts/run_logged.sh 11_verify -- python scripts/verify_training.py --adapter runs/main --dtype float32

## 6. `soup ship`, on the trained adapter and on the controls

A SHIP on the random control would show that a SHIP message cannot prove training worked.

In [ ]:
!bash scripts/run_logged.sh 12_ship_main -- soup ship --base Qwen/Qwen2.5-1.5B-Instruct --adapter runs/main --task-eval eval/task_eval_ru.jsonl --config configs/dpo_stream_t4.yaml --device cuda --emit-evidence runs/ship_main_evidence.json --output runs/ship_main_verdict.json

In [ ]:
!bash scripts/run_logged.sh 13_ship_control_random -- soup ship --base Qwen/Qwen2.5-1.5B-Instruct --adapter runs/verify_controls/control_random --task-eval eval/task_eval_ru.jsonl --config configs/dpo_stream_t4.yaml --device cuda --emit-evidence runs/ship_random_evidence.json --output runs/ship_random_verdict.json

## 7. Optional control run: the catalog recipes' learning rate

Same config, same data, same seed; only `lr` changes to the 5e-6 that every Soup catalog DPO
recipe ships with (with LoRA). The question for Part 2: the loss will still fall a little; does
the check say the model learned?

In [ ]:
!nohup bash scripts/run_logged.sh 15_train_control_recipe_lr -- python scripts/train_with_mem.py --config configs/control_recipe_lr.yaml --yes > /dev/null 2>&1 &

In [ ]:
!tail -c 1500 logs/t4/15_train_control_recipe_lr.log; echo; cat logs/t4/15_train_control_recipe_lr.exit 2>/dev/null || echo "(still running)"

In [ ]:
!bash scripts/run_logged.sh 16_verify_control_recipe_lr -- python scripts/verify_training.py --adapter runs/control_recipe_lr --dtype float32 --no-controls

## 8. Optional: `soup ship` on the null control

The null adapter adds exact zeros, so base and tuned should decode identically and leg 1 should
tie (DON'T SHIP). Run it only if the session has time left; it confirms the code reading.

In [ ]:
!bash scripts/run_logged.sh 14_ship_control_null -- soup ship --base Qwen/Qwen2.5-1.5B-Instruct --adapter runs/verify_controls/control_null --task-eval eval/task_eval_ru.jsonl --config configs/dpo_stream_t4.yaml --device cuda --emit-evidence runs/ship_null_evidence.json --output runs/ship_null_verdict.json

## 9. Package everything for the repo

In [ ]:
!cp nvidia-smi/00_start.txt nvidia-smi/99_end.txt 2>/dev/null; nvidia-smi > nvidia-smi/99_end.txt
!zip -q -r t4_outputs.zip logs nvidia-smi runs/*.json runs/main/adapter_config.json runs/main/adapter_model.safetensors runs/main/ref runs/control_recipe_lr/adapter_config.json runs/control_recipe_lr/adapter_model.safetensors -x "*/checkpoint-*"
from google.colab import files
files.download("t4_outputs.zip")